# 01 · Qwen2.5-VL-3B LoRA (원본 실험: EXP-001 → EXP-005 → EXP-007)

최종 앙상블의 **3B 모델**을 처음부터 재현합니다. 원본 코드는 `수인/EXP-001_masking_valid_seed43.ipynb` 셀 9~20, 36~38, 56~63이며, 하이퍼파라미터와 데이터 추출 방식을 그대로 옮겼습니다.

| 단계 | 원본 | 데이터 | 학습률 | 비고 |
|---|---|---|---|---|
| 1 | EXP-001 | train_split에서 200개 (seed 43) | 1e-4 | GradScaler + bf16 autocast |
| 2 | EXP-005 | 정답별 200개씩 추가 800개 | 5e-5 | grad clip 1.0 |
| 3 | EXP-007 | 남은 5,042개 | 2e-5 | grad clip 1.0 |
| 추론 | EXP-007 | valid 672 / test 6,714 | – | 해상도 1280×1280 픽셀 |

- 학습 이미지: 384×384 픽셀, 4bit(nf4) 양자화, LoRA r=8 α=16 (언어 모델 q/k/v/o/gate/up/down)
- loss: 정답 글자 토큰에만 적용 (assistant 응답 구간만 라벨)
- 원본 valid 결과: **639/672 (95.09%)**
- 권장 GPU: A100 (T4도 가능하나 느림)

In [ ]:
# ===== 0. 패키지 설치 (실행 후 '런타임 → 세션 다시 시작') =====
!pip install -q -U transformers accelerate peft bitsandbytes
!pip uninstall -y -q torchao          # 구버전 torchao가 peft와 충돌
!apt-get install -y -qq p7zip-full > /dev/null

In [ ]:
# ===== 1. 드라이브 연결 · 데이터 준비 =====
import os, glob, shutil, subprocess
from google.colab import drive
drive.mount("/content/drive")

SHARE_DIR = "/content/drive/MyDrive/ssafy/ai-challenge"   # 공유 폴더
OUT_DIR   = f"{SHARE_DIR}/repro"                           # 재현 결과 저장 위치 (원본 실험 파일을 덮어쓰지 않음)
os.makedirs(f"{OUT_DIR}/models", exist_ok=True)
DATA_ROOT = "/content/data"

cnt = lambda d: len(glob.glob(f"{DATA_ROOT}/{d}/*.jpg"))
if cnt("train") < 6714 or cnt("test") < 6714:
    # 드라이브에서 바로 풀면 실패할 수 있어 로컬로 복사 후 해제
    ZIP_SRC = glob.glob(f"{SHARE_DIR}/ssafy-16-2-ai*.zip")[0]
    shutil.copy(ZIP_SRC, "/content/data.zip")
    shutil.rmtree(DATA_ROOT, ignore_errors=True); os.makedirs(DATA_ROOT)
    subprocess.run(["7z","x","-y","-pocrisnotenough",f"-o{DATA_ROOT}","/content/data.zip"], check=True, stdout=subprocess.DEVNULL)
print("이미지 수 → train:", cnt("train"), "dev:", cnt("dev"), "test:", cnt("test"))
os.chdir(DATA_ROOT)   # CSV의 path 열(train/xxx.jpg)이 이 위치 기준

In [ ]:
# ===== 2. 설정 · 데이터 분할 (원본 EXP-001 셀 9, EXP-005 셀 36, EXP-007 셀 57) =====
import re, math, random
import numpy as np, pandas as pd, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any
from tqdm.auto import tqdm
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

Image.MAX_IMAGE_PIXELS = None
device = "cuda"
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGE_SIZE = 384
SEED = 43
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

train_pool = pd.read_csv(f"{SHARE_DIR}/split/train_split_seed43.csv")   # 6,042개
valid_df   = pd.read_csv(f"{SHARE_DIR}/split/valid_split_seed43.csv").reset_index(drop=True)   # 672개
test_df    = pd.read_csv(f"{DATA_ROOT}/test.csv").reset_index(drop=True)   # 6,714개

# 단계 1: 200개
initial_200 = train_pool.sample(n=200, random_state=SEED).reset_index(drop=True)
# 단계 2: 나머지에서 정답별 200개씩 = 800개
pool_after_200 = train_pool[~train_pool["id"].isin(initial_200["id"])].reset_index(drop=True)
additional_800 = (pool_after_200.groupby("answer", group_keys=False).sample(n=200, random_state=SEED)
                  .sample(frac=1, random_state=SEED).reset_index(drop=True))
# 단계 3: 남은 5,042개
used = set(initial_200["id"]) | set(additional_800["id"])
remaining_train = train_pool[~train_pool["id"].isin(used)].sample(frac=1, random_state=SEED).reset_index(drop=True)

print("단계별 학습 데이터:", len(initial_200), len(additional_800), len(remaining_train), "| valid", len(valid_df), "| test", len(test_df))
assert len(initial_200)+len(additional_800)+len(remaining_train) == 6042
assert not (set(train_pool["id"]) & set(valid_df["id"]))

In [ ]:
# ===== 3. 프롬프트 · Dataset · 정답 토큰 Loss Collator (원본 EXP-007 셀 57~58) =====
SYSTEM_INSTRUCT = ("You are a helpful visual question answering assistant. "
                   "Answer using exactly one letter among a, b, c, or d. No explanation.")

def build_mc_prompt(question, a, b, c, d):
    return (f"{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")

class VQAMCDataset(Dataset):
    def __init__(self, dataframe, processor, train=True):
        self.df = dataframe.reset_index(drop=True); self.processor = processor; self.train = train
    def __len__(self): return len(self.df)
    def __getitem__(self, index):
        row = self.df.iloc[index]
        with Image.open(row["path"]) as src: image = src.convert("RGB").copy()
        user_text = build_mc_prompt(str(row["question"]), str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"]))
        messages = [{"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                    {"role":"user","content":[{"type":"image","image":image},{"type":"text","text":user_text}]}]
        if self.train:
            messages.append({"role":"assistant","content":[{"type":"text","text":str(row["answer"]).strip().lower()}]})
        return {"messages": messages, "image": image}

def find_last_subsequence(sequence, sub):
    last = -1
    for p in range(len(sequence)-len(sub)+1):
        if sequence[p:p+len(sub)] == sub: last = p
    return last

@dataclass
class DataCollator:
    processor: Any
    train: bool = True
    def __call__(self, batch):
        texts = [self.processor.apply_chat_template(s["messages"], tokenize=False, add_generation_prompt=False) for s in batch]
        enc = self.processor(text=texts, images=[s["image"] for s in batch], padding=True, return_tensors="pt")
        if self.train:
            ids = enc["input_ids"]; labels = torch.full_like(ids, -100)
            pre = self.processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
            end = self.processor.tokenizer.encode("<|im_end|>", add_special_tokens=False)
            for b in range(ids.size(0)):
                seq = ids[b].tolist()
                p = find_last_subsequence(seq, pre); assert p != -1, "assistant 시작 토큰 없음"
                s = p + len(pre)
                e = find_last_subsequence(seq[s:], end); assert e != -1, "assistant 종료 토큰 없음"
                labels[b, s:s+e] = ids[b, s:s+e]          # 정답 글자에만 loss
            enc["labels"] = labels
        return enc

def make_loader(df):
    g = torch.Generator(); g.manual_seed(SEED)
    return DataLoader(VQAMCDataset(df, processor, True), batch_size=1, shuffle=True,
                      collate_fn=DataCollator(processor, True), num_workers=0, pin_memory=True, generator=g)

In [ ]:
# ===== 4. 모델 (원본 EXP-001 셀 11) =====
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=IMAGE_SIZE*IMAGE_SIZE, max_pixels=IMAGE_SIZE*IMAGE_SIZE, trust_remote_code=True)
processor.tokenizer.padding_side = "right"
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()
lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                         target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# Loss 대상 확인: 정답 글자 1토큰이어야 함
chk = DataCollator(processor, True)([VQAMCDataset(initial_200.head(2), processor, True)[i] for i in range(2)])
for i in range(2):
    t = chk["labels"][i][chk["labels"][i] != -100]
    print("정답", initial_200.iloc[i]["answer"], "→ Loss 대상", repr(processor.tokenizer.decode(t)))

In [ ]:
# ===== 5. 학습 함수 (세 단계의 원본 루프 차이를 옵션으로 보존) =====
def train_stage(df, lr, name, use_scaler, clip, divide_by_group):
    """use_scaler: 단계1(GradScaler 사용) / clip: 단계2·3(grad clip 1.0) / divide_by_group: 단계1·3(마지막 묶음은 실제 크기로 나눔)"""
    GRAD_ACCUM = 4
    loader = make_loader(df)
    model.gradient_checkpointing_enable(); model.enable_input_require_grads()
    model.config.use_cache = False; model.train()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr)
    total = math.ceil(len(loader)/GRAD_ACCUM)
    sched = get_linear_schedule_with_warmup(opt, max(1, int(total*0.03)), total)
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    opt.zero_grad(set_to_none=True); upd = 0; run = 0.0; n = 0
    bar = tqdm(loader, desc=name)
    for step, batch in enumerate(bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        gstart = ((step-1)//GRAD_ACCUM)*GRAD_ACCUM
        gsize = min(GRAD_ACCUM, len(loader)-gstart) if divide_by_group else GRAD_ACCUM
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            raw = model(**batch).loss; loss = raw / gsize
        scaler.scale(loss).backward() if use_scaler else loss.backward()
        run += raw.item(); n += 1
        if step % GRAD_ACCUM == 0 or step == len(loader):
            if use_scaler:
                scaler.step(opt); scaler.update()
            else:
                if clip: torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
                opt.step()
            sched.step(); opt.zero_grad(set_to_none=True); upd += 1
            bar.set_postfix(loss=f"{run/n:.4f}", lr=f"{sched.get_last_lr()[0]:.2e}"); run = 0.0; n = 0
    save = f"{OUT_DIR}/models/{name}"
    model.save_pretrained(save); processor.save_pretrained(save)
    print(f"{name} 완료 | 업데이트 {upd} | 저장 {save}")

In [ ]:
# ===== 6. 3단계 학습 (A100 기준 약 1.5~2시간) =====
train_stage(initial_200,     lr=1e-4, name="EXP-001_3b_200",  use_scaler=True,  clip=False, divide_by_group=True)   # 원본 셀 20
train_stage(additional_800,  lr=5e-5, name="EXP-005_3b_1000", use_scaler=False, clip=True,  divide_by_group=False)  # 원본 셀 38
train_stage(remaining_train, lr=2e-5, name="EXP-007_3b_6042", use_scaler=False, clip=True,  divide_by_group=True)   # 원본 셀 59

In [ ]:
# ===== 7. 1280 해상도 추론 (원본 EXP-007 셀 60~63) =====
model.gradient_checkpointing_disable(); model.config.use_cache = True; model.eval()
CHOICES = ["a","b","c","d"]
processor_1280 = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=1280*1280, max_pixels=1280*1280, trust_remote_code=True)
processor_1280.tokenizer.padding_side = "left"
choice_ids = [processor_1280.tokenizer.encode(c, add_special_tokens=False)[0] for c in CHOICES]

def predict(df, desc, batch_size=1):
    probs = []
    for s in tqdm(range(0, len(df), batch_size), desc=desc):
        texts, imgs = [], []
        for _, row in df.iloc[s:s+batch_size].iterrows():
            with Image.open(row["path"]) as src: image = src.convert("RGB").copy()
            msgs = [{"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                    {"role":"user","content":[{"type":"image","image":image},
                     {"type":"text","text":build_mc_prompt(str(row["question"]),str(row["a"]),str(row["b"]),str(row["c"]),str(row["d"]))}]}]
            texts.append(processor_1280.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)); imgs.append(image)
        inp = {k: v.to(device) for k, v in processor_1280(text=texts, images=imgs, padding=True, return_tensors="pt").items()}
        with torch.inference_mode(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(**inp).logits[:, -1, :][:, choice_ids]
            probs += torch.softmax(logits.float(), dim=-1).cpu().tolist()
        del inp; torch.cuda.empty_cache()
    return np.array(probs)

pv = predict(valid_df, "3B valid 1280")
vres = valid_df.copy(); vres["pred"] = [CHOICES[i] for i in pv.argmax(1)]; vres["correct"] = vres["pred"] == vres["answer"]
for i, c in enumerate(CHOICES): vres[f"prob_{c}"] = pv[:, i]
vres.to_csv(f"{OUT_DIR}/3b_valid_probs.csv", index=False, encoding="utf-8-sig")
print(f"3B valid: {vres.correct.mean():.4f} ({vres.correct.sum()}/672)   ※ 원본 EXP-007: 0.9509 (639/672)")

pt = predict(test_df, "3B test 1280")
tres = test_df[["id"]].copy()
for i, c in enumerate(CHOICES): tres[f"prob_{c}"] = pt[:, i]
tres.to_csv(f"{OUT_DIR}/3b_test_probs.csv", index=False, encoding="utf-8-sig")
print("저장:", f"{OUT_DIR}/3b_test_probs.csv")